# Phase 1: one topic in, one brief out

Runs the phase 1 graph, `planner -> researcher -> writer`, on one real topic, and keeps the record of the run:

1. the brief, rendered below and saved to `notebooks/phase_01_brief.md`;
2. the run numbers (sub-questions, findings, sections, retries, time);
3. the link to the run's trace in LangSmith.

Needs real keys in `.env`. One run costs up to 8 model calls and 6 Tavily searches, and takes about a minute.

Outputs are stripped from this file when it is committed (`nbstripout`); the brief file is the committed record. Concept: `tracing-from-day-one`.

In [ ]:
# Work from the project root, wherever Jupyter started: settings.py reads .env
# from the working folder, and the graph package is imported from the root.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT.name}")

In [ ]:
import logging
import re
import time
from uuid import uuid4

from IPython.display import Markdown, display
from langchain_core.tracers.langchain import wait_for_all_tracers

from graph.llm import ModelGateway
from graph.nodes.researcher import RESULTS_PER_SUB_QUESTION, TavilySearchClient
from graph.prompts import planner as planner_prompt
from graph.prompts import writer as writer_prompts
from graph.report import cited_finding_ids, render_brief
from graph.supervisor import build_graph, research_run_config
from settings import settings


class RunLogRecorder(logging.Handler):
    """Keeps the graph's log messages, so retries and removed markers can be counted."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.messages.append(record.getMessage())


# Every module under graph/ logs to a child of the "graph" logger.
graph_logger = logging.getLogger("graph")
graph_logger.setLevel(logging.INFO)
for old_recorder in [h for h in graph_logger.handlers if isinstance(h, RunLogRecorder)]:
    graph_logger.removeHandler(old_recorder)  # re-running this cell must not count twice
run_log = RunLogRecorder()
graph_logger.addHandler(run_log)

In [ ]:
TOPIC = "Due diligence on Symbotic: business model, customers, and main risks"

graph = build_graph(ModelGateway(), TavilySearchClient())
# CONCEPT: tracing-from-day-one. The run id is chosen before the run, so the
# trace can be looked up afterwards (cell below).
run_id = uuid4()
run_log.messages.clear()
started_at = time.perf_counter()

final_state = await graph.ainvoke(
    {"topic": TOPIC, "document_ids": []},
    config=research_run_config(TOPIC, run_id),
)

elapsed_seconds = time.perf_counter() - started_at
print(f"Done in {elapsed_seconds:.1f} s")

In [ ]:
brief_markdown = render_brief(TOPIC, final_state["draft"], final_state["findings"])
display(Markdown(brief_markdown))

In [ ]:
def count_messages(fragment: str) -> int:
    return sum(fragment in message for message in run_log.messages)


def count_multi_id_markers() -> int:
    counts = re.findall(r"markers citing more than one finding: (\d+)", " ".join(run_log.messages))
    return sum(int(count) for count in counts)


findings = final_state["findings"]
fetched_ids = {finding.id for finding in findings if finding.found}
run_numbers = {
    "sub-questions": len(final_state["sub_questions"]),
    "findings found": len(fetched_ids),
    "no evidence": sum(not finding.found for finding in findings),
    "sources cited": len(cited_finding_ids(final_state["draft"]) & fetched_ids),
    "sections": len(final_state["draft"]),
    "sections added for missed sub-questions": count_messages("outline missed"),
    "unknown markers removed": count_messages("removed unknown markers"),
    "multi-id markers": count_multi_id_markers(),
    "schema retries": count_messages("did not fit"),
    "transient retries": count_messages("transient error"),
    "seconds": round(elapsed_seconds, 1),
}
for measure, value in run_numbers.items():
    print(f"{measure:<42}{value}")

In [ ]:
# CONCEPT: tracing-from-day-one. LangSmith uploads runs from a background
# thread; wait until everything is sent, then find this run by its id.
wait_for_all_tracers()

if settings.langsmith_tracing:
    from langsmith import Client

    langsmith_client = Client()
    for lookup_attempt in range(1, 6):
        try:
            traced_run = langsmith_client.read_run(run_id)
            trace_url = langsmith_client.get_run_url(
                run=traced_run, project_name=settings.langsmith_project
            )
            print(f"Trace: {trace_url}")
            break
        except Exception:  # a new run can take a few seconds to appear
            time.sleep(2 * lookup_attempt)
    else:
        print(f"Trace not found yet: search project {settings.langsmith_project} for {run_id}")
else:
    print("Tracing is off (LANGSMITH_TRACING=false)")

In [ ]:
# The committed record of the run. Only the brief and its provenance are
# written: never settings, keys, or LangSmith ids (option C, three layers).
prompt_versions = (
    f"P-01 {planner_prompt.VERSION}, P-03 {writer_prompts.OUTLINE_VERSION}, "
    f"P-04 {writer_prompts.SECTION_VERSION}"
)
provenance_lines = [
    "> - Generated by `notebooks/phase_01.ipynb`, Phase 1.",
    f"> - Prompts: {prompt_versions}.",
    f"> - Models: `{settings.vre_chat_model}` (planner), `{settings.vre_writer_model}` (writer).",
    f"> - Search: Tavily, {RESULTS_PER_SUB_QUESTION} results per sub-question.",
    f"> - Run: {run_numbers['sub-questions']} sub-questions, "
    f"{run_numbers['findings found']} findings, {run_numbers['sections']} sections, "
    f"{run_numbers['seconds']} s.",
    "> - Phase 1: each citation names the passage its sentence was written from, but citations "
    "are not yet verified. Claim checking arrives in phase 5.",
]
title_line, brief_body = brief_markdown.split("\n", 1)
brief_file_text = "\n".join([title_line, "", *provenance_lines, brief_body])

brief_path = PROJECT_ROOT / "notebooks" / "phase_01_brief.md"
brief_path.write_text(brief_file_text, encoding="utf-8")
print(f"Wrote {brief_path.relative_to(PROJECT_ROOT)} ({len(brief_file_text.splitlines())} lines)")

## Problems we hit and what we decided

The phase record, written after the phase from the decision log, the troubleshooting log, and the concept notes. One entry per problem met while building Phase 1: what happened, what caused it (or what a check showed when the first guess was wrong), the options, and the decision. Decision numbers refer to the project's decision log.

### 1. A file pasted into the editor arrived cut in half

**What happened.** `ruff` reported unused imports and an undefined function in `graph/llm.py`, and the live gateway test failed with `AttributeError: 'ModelGateway' object has no attribute '_invoke_with_transport_retry'`.
**Cause.** Only the first half of the 208-line file reached the editor; the code was fine.
**Decision.** Files longer than about 100 lines are sent as downloads, and the line count is compared after copying. **Lesson:** when the errors make no sense for the code, check that the file is the code.

### 2. A new download was replaced by an older copy

**What happened.** The new tests could not import `FakeSearchClient`, and the copied `conftest.py` had 118 lines instead of 144.
**Cause.** The browser saved the new file as `conftest (1).py`, because an older `conftest.py` was still in the downloads folder, and the copy command took the old one.
**Decision.** A small script, `take.ps1`, places every file: it takes the newest of `name.ext`, `name (1).ext`, and so on, prints its line count, and clears the downloads. Every file since has been placed with it.

### 3. A check that never fired on real data (D-29)

**What happened.** The first live brief contained `[Q4-F1, Q4-F2]`. The writer's marker check reported nothing removed, and every test passed.
**Cause.** The check's pattern matched one id inside brackets, so a list of ids did not match at all: those markers were never checked. Every scripted reply used single-id markers, so the tests proved the check only on the cases they contained.
**Decision.** The pattern accepts comma-separated ids and checks each one; an unknown id is dropped and an empty marker removed (D-29). A new test uses exactly the case from the live run and fails on the old code. **Lesson:** a check that reports "0 removed" on real data has either found nothing wrong or is not looking. Reading the live output is part of testing.

### 4. The secrets test raised a false alarm on its first real file

**What happened.** The test that keeps keys out of the repository flagged `.env.example`, a template with every secret empty.
**Cause.** The pattern allowed any whitespace after `=`, and a line break counts as whitespace, so the empty `LANGSMITH_API_KEY=` ran on into the next line. A second false alarm came while fixing it: the new test's own fake key, written out as text, flagged the test file itself.
**Decision.** Spaces and tabs only around `=`; a test of the patterns themselves against a full empty template; fake keys built at run time. Separately, a placeholder key as short as `ci` made a "key not printed" test fail by chance, so placeholder keys stay long and distinctive. **Lesson:** a scanner needs tests too, run against the real files it will scan.

### 5. A schema makes a plan valid, not good

**What happened.** On the first two live runs, every plan passed the schema, the cap, and the route checks, yet read by hand: sub-questions joined two questions with "and", a topic about the future produced "trends in 2023-2024", and both plans had exactly six questions.
**Options.** Rewrite the planner prompt at once; record the weak points and tune the prompt when plans can be scored.
**Decision.** Recorded as P-01's weak points, for a second version once Phase 10's evaluation can measure a change. Two runs are too few to tell a better prompt from luck. **Lesson:** structured output guarantees the shape of an answer, not its quality.

### 6. The history did not say what each commit was for (D-23, D-27)

**What happened.** The first code commits had plain subjects, so the log could not show which phase or concept a change belonged to; and the order of work was still open: every concept's notes first, or one concept at a time.
**Options.** Phase only in tags; conventional `feat:` and `chore:` prefixes; a phase-and-concept prefix.
**Decision.** Every commit subject starts with `[P<phase> <concept-slug>]`, the same name as the concept's notes and its `# CONCEPT:` comments, and each phase ends with a tag (D-27); the first 11 commits were renamed while the repository was new. Concepts are built one at a time: introduce, document, code, test, then the next (D-23).

### 7. The repository's front page described another project

**What happened.** The README described a different agent and showed broken characters where emojis had been.
**Cause.** It had been filled from another project when the repository was created, and nobody had read it since.
**Decision.** A new README that claims only what works at the end of Phase 1, and a standing step: read the README before closing each phase.

### 8. The tools got in the way

**What happened.** Three problems outside the code: `git commit` found nothing to commit and pytest collected no tests; the terminal activated another project's Python environment; and pushes failed with SSL errors, then "Could not resolve host".
**Cause.** Files open in the editor but not saved; a global editor setting pointing at an old environment; the home network failing to resolve GitHub for a while.
**Decision.** Save all before committing; clear the global setting and check that the prompt names this project's environment before running anything; commit locally and push once the name resolves again (`ipconfig /flushdns` cleared it). Each is in the troubleshooting log.